# Tableau Dashboard Prep

**Week 8: Dashboard (Tableau) & Final Packaging**

Prepare all the data Tableau needs to build the final dashboard. Two things happen here:
1. Score the **entire** population (train + test combined) with the final XGBoost model, and build a single portfolio-level dataset with segments (age group, income group) ready for slicing in Tableau.
2. Gather the model validation artifacts already produced in earlier notebooks (Model A vs B comparison, calibration table, cutoff analysis) and copy everything into a dedicated `tableau_data/` folder Tableau can connect to directly.


## 1. Build the Portfolio-Level Dataset

Score the full population (not just the test set) so the dashboard reflects every applicant, not only the held-out evaluation sample. Predicted probability of default is converted into a FICO-like `PREDICTED_RISK_SCORE` (0-850, higher = safer), and two segment columns (`AGE_GROUP`, `INCOME_GROUP`) are added so Tableau can break down default rate and risk score by segment without needing calculated fields.


In [2]:
import pandas as pd
import numpy as np
import joblib

# Load all the final pieces (not from master_features.csv)
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()
train_ids = pd.read_csv('../data/processed/train_ids.csv').squeeze()
test_ids = pd.read_csv('../data/processed/test_ids.csv').squeeze()

# Recombine into one full dataset (full population, not just test)
X_full = pd.concat([X_train, X_test], ignore_index=True)
y_full = pd.concat([y_train, y_test], ignore_index=True)
ids_full = pd.concat([train_ids, test_ids], ignore_index=True)

cat_cols = X_full.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    X_full[col] = X_full[col].astype('category')

# Load model, predict directly (X_full is already in its final structure, nothing left to drop)
xgb_model = joblib.load('../models/xgboost_final.pkl')
prob_default = xgb_model.predict_proba(X_full)[:, 1]

# Build the combined result dataframe
df_tableau = X_full.copy()
df_tableau['SK_ID_CURR'] = ids_full.values
df_tableau['TARGET'] = y_full.values
df_tableau['PREDICTED_PROB_DEFAULT'] = prob_default
df_tableau['PREDICTED_RISK_SCORE'] = ((1 - df_tableau['PREDICTED_PROB_DEFAULT']) * 850).round(0)

# Segmentation for the dashboard
df_tableau['AGE_GROUP'] = pd.cut(df_tableau['AGE_YEARS'],
                                   bins=[0, 25, 35, 45, 55, 100],
                                   labels=['<25', '25-35', '35-45', '45-55', '55+'])

df_tableau['INCOME_GROUP'] = pd.qcut(df_tableau['AMT_INCOME_TOTAL'],
                                       q=5,
                                       labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

tableau_cols = [
    'SK_ID_CURR', 'TARGET', 'PREDICTED_PROB_DEFAULT', 'PREDICTED_RISK_SCORE',
    'IS_ANOMALY', 'ANOMALY_SCORE',
    'AGE_YEARS', 'AGE_GROUP', 'AMT_INCOME_TOTAL', 'INCOME_GROUP',
    'AMT_CREDIT', 'AMT_ANNUITY', 'NAME_CONTRACT_TYPE', 'CODE_GENDER',
    'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'REGION_RATING_CLIENT_W_CITY',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'BUREAU_MEAN_DAYS_CREDIT', 'INST_RATIO_LATE', 'CC_MEAN_UTILIZATION',
    'PREV_RATIO_REFUSED'
]

df_export = df_tableau[tableau_cols].copy()
df_export.to_csv('../data/processed/tableau_portfolio_data.csv', index=False)
print(df_export.shape)
df_export.head()

(307511, 27)


,SK_ID_CURR,TARGET,PREDICTED_PROB_DEFAULT,PREDICTED_RISK_SCORE,IS_ANOMALY,ANOMALY_SCORE,AGE_YEARS,AGE_GROUP,AMT_INCOME_TOTAL,INCOME_GROUP,...,OCCUPATION_TYPE,ORGANIZATION_TYPE,REGION_RATING_CLIENT_W_CITY,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,BUREAU_MEAN_DAYS_CREDIT,INST_RATIO_LATE,CC_MEAN_UTILIZATION,PREV_RATIO_REFUSED
0,310536,0,0.308006,588.0,0,0.182216,33.693151,25-35,90000.0,Very Low,...,Laborers,Business Entity Type 2,2,0.384582,0.289573,0.622922,-1175.800000,0.0,0.346744,0.000000
1,365516,0,0.448362,469.0,0,0.171592,42.123288,35-45,90000.0,Very Low,...,Drivers,Other,3,0.505998,0.514261,0.535276,0.000000,0.0,0.000000,0.166667
2,242055,1,0.463679,456.0,0,0.191353,52.895890,45-55,135000.0,Low,...,Laborers,Business Entity Type 3,2,0.505998,0.486906,0.598926,-1358.500000,0.0,0.000000,0.000000
3,454894,1,0.440093,476.0,0,0.176284,48.742466,45-55,135000.0,Low,...,Laborers,Business Entity Type 3,2,0.505998,0.675705,0.454321,-2005.333333,0.0,0.000000,0.250000
4,448321,0,0.695819,259.0,0,0.164787,23.249315,<25,180000.0,High,...,Accountants,Restaurant,2,0.505998,0.154565,0.535276,0.000000,0.0,0.000000,0.500000


**Insight:** the exported dataset has 307,511 rows (the full population) and 27 columns — a mix of IDs, the model outputs (`PREDICTED_PROB_DEFAULT`, `PREDICTED_RISK_SCORE`), the anomaly detection flag/score, demographic/segment fields, and the top predictive features from earlier notebooks (`EXT_SOURCE_*`, `BUREAU_MEAN_DAYS_CREDIT`, `INST_RATIO_LATE`, `CC_MEAN_UTILIZATION`, `PREV_RATIO_REFUSED`). This is deliberately a flat, wide table — the format Tableau works with best.


## 2. Verify the Export


In [3]:
print("File saved to: data/processed/tableau_portfolio_data.csv")
print(f"Total rows: {len(df_export)}")
print(f"Missing values: {df_export.isnull().sum().sum()}")

File saved to: data/processed/tableau_portfolio_data.csv
Total rows: 307511
Missing values: 0


**Insight:** 307,511 rows exported with 0 missing values — confirms the full-population scoring pipeline ran cleanly end-to-end and the file is ready to hand off to Tableau without any additional cleaning.


## 3. Load Model Validation Artifacts

Pull back in the model comparison table, the score calibration table, and the cutoff/approval-rate analysis — all previously saved in the Model A notebook — so they can be bundled alongside the portfolio dataset.


In [4]:
# Load the results we saved previously
model_comparison = pd.read_csv('../data/processed/model_comparison.csv')
calib_table = pd.read_csv('../data/processed/model_a_calibration.csv')
cutoff_df = pd.read_csv('../data/processed/model_a_cutoff_analysis.csv')

print("=== Model Comparison ===")
print(model_comparison)
print("\n=== Calibration Table ===")
print(calib_table)
print("\n=== Cutoff Analysis ===")
print(cutoff_df)

=== Model Comparison ===
                  Metric Model A (Logistic Regression + WOE) Model B (XGBoost)
0               Test AUC                              0.7672            0.7742
1                Test KS                                 0.4             0.412
2     Train-Test AUC Gap                              0.0001             0.019
3        Active Features                                  62               132
4  Explainability Method           Native (Scorecard Points)   SHAP (Post-hoc)

=== Calibration Table ===
         score_bucket  count  actual_default_rate  mean_score
0  (571.326, 643.585]   6151             0.279142  629.101534
1  (643.585, 657.203]   6150             0.151382  650.836402
2   (657.203, 667.22]   6150             0.100650  662.415205
3   (667.22, 675.624]   6150             0.083740  671.513812
4   (675.624, 683.47]   6151             0.062591  679.604167
5   (683.47, 691.077]   6150             0.042764  687.309977
6  (691.077, 699.129]   6150            

**Insight:** these three tables map directly to dashboard sheets: `model_comparison` powers a Model A vs B performance sheet, `calib_table` powers the calibration/reliability chart (score bucket vs actual default rate), and `cutoff_df` powers the approval-rate-vs-default-rate trade-off chart used for cutoff decision-making.


## 4. Copy Files into the Tableau Data Folder

Consolidate the portfolio dataset and the three validation tables into a single `tableau_data/` folder, so Tableau can point at one clean directory instead of reaching into `data/processed/`.


In [5]:
import os
import shutil

os.makedirs('../tableau_data', exist_ok=True)

files_to_copy = [
    'tableau_portfolio_data.csv',
    'model_comparison.csv',
    'model_a_calibration.csv',
    'model_a_cutoff_analysis.csv'
]

for f in files_to_copy:
    shutil.copy(f'../data/processed/{f}', f'../tableau_data/{f}')

print("Files copied to tableau_data/")
print(os.listdir('../tableau_data'))

Files copied to tableau_data/
['model_a_calibration.csv', 'model_a_cutoff_analysis.csv', 'model_comparison.csv', 'tableau_portfolio_data.csv']


**Insight:** all 4 files (`tableau_portfolio_data.csv`, `model_comparison.csv`, `model_a_calibration.csv`, `model_a_cutoff_analysis.csv`) are now in one place — this is the complete data foundation for the 4-sheet Tableau dashboard: Portfolio Overview, Anomaly Detection Summary, Model Performance, and Cut-off & Monitoring.


---
### Summary & Next Steps
- Scored the full population (307,511 applicants) with the final XGBoost model and built a single wide portfolio table with a FICO-like risk score (0-850) plus age/income segments — ready for Tableau.
- Verified the export: 307,511 rows, 0 missing values.
- Gathered the Model A vs B comparison, calibration table, and cutoff analysis from earlier notebooks.
- Copied everything into `tableau_data/`, the single source folder for the dashboard.
- Next: build the 4 Tableau sheets (Portfolio Overview, Anomaly Detection Summary, Model Performance, Cut-off & Monitoring), then finalize the GitHub repo (README, architecture diagram) and write the LinkedIn post.
